In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS shopsphere.quarantine;
CREATE SCHEMA IF NOT EXISTS shopsphere.silver;

CREATE TABLE IF NOT EXISTS shopsphere.silver.orders(
    order_id int,
    customer_id int,
    order_date timestamp,
    order_status string,
    shipping_address string,
    total_amount decimal(10,2),
    created_at timestamp,
    updated_at timestamp
)
USING DELTA;

CREATE TABLE IF NOT EXISTS shopsphere.quarantine.orders(
    order_id int,
    customer_id int,
    order_date timestamp,
    order_status string,
    shipping_address string,
    total_amount decimal(10,2),
    created_at timestamp,
    updated_at timestamp
)
USING DELTA;

In [0]:
from pyspark.sql.functions import *
from delta.tables import *

df = spark.read.table("shopsphere.bronze.orders")

In [0]:
#drop duplicates
df_dodup = df.dropDuplicates()

In [0]:
#validate customer_id
df_valid = df_dodup.filter(col("customer_id").isNotNull())

df_quarantine = df_dodup.filter(col("customer_id").isNotNull()==False
                                ).withColumn("validation_status", lit("invalid_customer_id"))



In [0]:
#quarantine rejected rows from customer_id
quarantine_table = DeltaTable.forName(spark,"shopsphere.quarantine.orders")

quarantine_table.alias("target").merge(df_quarantine.alias("source"),
                                       "target.order_id = source.order_id").\
                                        whenNotMatchedInsertAll().\
                                        withSchemaEvolution().\
                                        execute()

In [0]:
#Standardize Order status
df_order_status = df_valid.withColumn("order_status", initcap(trim(col("order_status"))))

In [0]:
#Validate order dates
df_valid_date = df_order_status.filter(
    col("order_date").isNotNull() &
    (col("order_date") <= current_timestamp()))

df_quarantine = df_order_status.filter(
    col("order_date").isNull() | (col("order_date") > current_timestamp())
    ).withColumn("validation_status", lit("invalid_order_date"))

#quarantine rejected rows from order_date
quarantine_table = DeltaTable.forName(spark,"shopsphere.quarantine.orders")

quarantine_table.alias("target").merge(df_quarantine.alias("source"),
                                       "target.order_id = source.order_id").\
                                        whenNotMatchedInsertAll().\
                                        withSchemaEvolution().\
                                        execute()

In [0]:
#validate negative order amounts
df_valid_amount = df_valid_date.filter(col("total_amount")>=0)

df_quarantine = df_valid_date.filter(col("total_amount")<0).withColumn("validation_status", lit("invalid_total_amount"))

#quarantine rejected rows from total_amount
quarantine_table = DeltaTable.forName(spark,"shopsphere.quarantine.orders")

quarantine_table.alias("target").merge(df_quarantine.alias("source"),
                                       "target.order_id = source.order_id").\
                                        whenNotMatchedInsertAll().\
                                        withSchemaEvolution().\
                                        execute()

In [0]:
#write transformed data to silver table
silver_table = DeltaTable.forName(spark,"shopsphere.silver.orders")

silver_table.alias("target").merge(df_valid_amount.alias("source"), \
                                       "target.order_id = source.order_id" ).\
                        whenNotMatchedInsertAll().\
                        whenMatchedUpdateAll().\
                        execute()

In [0]:
%sql

SELECT * FROM shopsphere.silver.orders
WHERE order_status in ("Cancelled","Complete","Returned")